In [1]:
# %load_ext autoreload
# %autoreload 2

In [1]:
import sys
import os
import argparse
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

sys.path.append("../../../")
import data_loading as dl
from importlib import reload
reload(dl)

# from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

from microfit import detsys

In [3]:
def make_error_plot(hist_generator: [hist.HistogramGenerator], output_file: str):
    for hist_gen in hist_generator:
        hist_all_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=True,
            #use_sideband=False,
            add_precomputed_detsys=True,
            #normalization_uncertainty=[0.01,0.02], # should be automatically passed through from the hist_generator
        )
        # The unisim errors are GENIE knobs, so we include them but only for the GENIE errors
        hist_genie_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=True,
            #use_sideband=False,
            ms_columns=["weightsGenie"],
            include_unisim_errors=True,
            include_stat_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=None,
        )
        hist_flux_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=True,
            #use_sideband=False,
            ms_columns=["weightsFlux"],
            include_unisim_errors=False,
            include_stat_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=None,
        )
        hist_reint_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=True,
            #use_sideband=False,
            ms_columns=["weightsReint"],
            include_unisim_errors=False,
            include_stat_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=None,
        )
        hist_stat_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=False,
            #use_sideband=False,
            include_stat_errors=True,
            include_unisim_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=None,
        )
        hist_detsys_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=False,
            #use_sideband=False,
            include_stat_errors=False,
            include_unisim_errors=False,
            add_precomputed_detsys=True,
            #normalization_uncertainty=None,
        )
        hist_POT_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=False,
            #use_sideband=False,
            include_unisim_errors=False,
            include_stat_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=[0.02]
        )
        hist_Ntargets_errors = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=False,
            #use_sideband=False,
            include_unisim_errors=False,
            include_stat_errors=False,
            add_precomputed_detsys=False,
            #normalization_uncertainty=[0.01]
        )
        hist_all_except_stats = hist_gen.generate_joint_histogram(
            hist_generators = hist_gen,
            include_multisim_errors=True,
            #use_sideband=False,
            include_unisim_errors=True,
            include_stat_errors=False,
            add_precomputed_detsys=True,
            #normalization_uncertainty=[0.01,0.02], # should be automatically passed through from the hist_generator
        )
    
    print(hist_generator.channels)
    for channel in hist_generator.channels:
        bin_counts = hist_all_errors[channel].bin_counts
        # Total error as fraction of bin count
        total_errors = hist_all_errors[channel].std_devs / bin_counts
        # Every error source as fraction of total error
        genie_errors = hist_genie_errors[channel].std_devs / hist_all_errors[channel].std_devs
        flux_errors = hist_flux_errors[channel].std_devs / hist_all_errors[channel].std_devs
        reint_errors = hist_reint_errors[channel].std_devs / hist_all_errors[channel].std_devs
        detsys_errors = hist_detsys_errors[channel].std_devs / hist_all_errors[channel].std_devs
        POT_errors = hist_POT_errors[channel].std_devs / hist_all_errors[channel].std_devs
        Ntargets_errors = hist_Ntargets_errors[channel].std_devs / hist_all_errors[channel].std_devs
        all_except_stat_errors = (
            hist_all_except_stats[channel].std_devs / hist_all_errors[channel].std_devs
        )
        stat_errors = hist_stat_errors[channel].std_devs / hist_all_errors[channel].std_devs

        bin_edges = hist_all_errors[channel].binning.bin_edges
        n_bins = len(bin_edges) - 1
        
        # Plotting the fractional errors
        
        plt.stairs(genie_errors, bin_edges, label='GENIE', linestyle='dashdot')
        plt.stairs(flux_errors, bin_edges, label='Flux', linestyle='dashdot')
        plt.stairs(reint_errors, bin_edges, label='Reinteractions', linestyle='dashdot')
        plt.stairs(detsys_errors, bin_edges, label='DetSyst', linestyle='dashdot')
        plt.stairs(POT_errors, bin_edges, label='POT', linestyle='dashdot')
        plt.stairs(Ntargets_errors, bin_edges, label='NTargets', linestyle='dashdot')
        plt.stairs(stat_errors, bin_edges, label='MC Stat', linestyle='dashdot')
        
        plt.stairs(all_except_stat_errors, bin_edges, label='Total Syst Errors', linestyle='dashdot', color='black')
        plt.stairs(total_errors, bin_edges, label='Total Errors (Syst + Stat)', linestyle='solid', color='black', lw=1.7)
        
        plt.legend()

In [4]:
keep_vars = [
    "Signal_1e1p", "mc_signal_1e1p", "nu_pdg", "TrueElecIdx", "TrueLeadProtonIdx", "InFV", "HasNoMesons",
    "TrueNElec", "TrueNProt", "TrueDeltaPT", "TrueDeltaAlphaT", "TruePN", "TrueAlpha3D",
    "nproton", "npion", "npi0", "nelec", "nmuon", "isVtxInFiducial", "ccnc",
    "Sel_1e1p", "sel_1e1p_w_cuts", "RecoElectronCandidateIdx", "RecoLeadProtonCandidateIdx", "InFV_reco",
    "RecoElecPassMomCut", "RecoLeadProtonPassMomCut", "n_reco_tracks", "n_reco_showers",
    "RecoDeltaPT", "RecoDeltaAlphaT", "RecoPN", "RecoAlpha3D", "RecoECal", "Reco_mag_q", "RecoPL",
    "nslice", "selected", "shr_energy_tot_cali", "_opfilter_pe_beam", "_opfilter_pe_veto", "bnbdata", "extdata",
    "CosmicIPAll3D", "hits_ratio", "shrmoliereavg", "subcluster", "trkfit", "tksh_distance",
    "shr_tkfit_nhits_tot", "shr_tkfit_dedx_max", "tksh_angle", "shr_trk_len"
]

# If needed, add TKI ingredients to this list

In [5]:
RUN = ["3"]
#RUN = ["1","2","3_nocrt","3_crt","4b","4c","4d","5"]
#RUN = ["1","2","3","4b","4c","4d","5"] #important that it's a string 1) new format to include latest runs 2) to include 'mc_pdg' otherwise it gets dropped
blinded = True

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=False,
    load_lee=False,
    load_nue_tki=True,
    keep_columns=keep_vars,
    blinded=blinded,
    load_crt_vars=False,
    enable_cache=True,
)

Loading run 3
Updating keep_columns with truth-filtering variables: {'nu_pdg', 'ccnc'}


In [6]:
selection = "OnePL_new"
preselection = "OneP_new"

detector_variations = ["cv","lydown"] #,"lyatt","lyrayleigh","sce","recomb2","wiremodx","wiremodyz","wiremodthetaxz","wiremodthetayz"]

for binning_def in vdef.TKI_variables_1e1p:
    # some binning definitions have more than 4 elements,
    # we ignore the last ones for now
    #binning = hist.Binning.from_config(*binning_def[:4])
    binning = hist.Binning.from_config(*binning_def)  # for variable bin sizes
    #print(binning_def)
    
    # Load detvars
    detvar_data = detsys.make_variations(
    run_numbers=RUN,
    data="bnb",
    binning=binning,
    selection=selection,
    preselection=preselection,
    use_kde_smoothing=False,
    make_plots=False,
    plot_output_dir= "/exp/uboone/app/users/mmoudgal/PELEE/sandbox/mmoudgalya/analysis_1e1p/analysis_plots/detsys/",
    enable_detvar_cache=True,
    detvar_cache_dir="/exp/uboone/data/users/mmoudgal/PELEE/detvar_cached_dataframes/",
    extra_selection_query=None,
    show_plots=True,
    variations=detector_variations,
    #**dl_kwargs,
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=False,
    load_lee=False,
    load_nue_tki=True,
    keep_columns=keep_vars,
    blinded=blinded,
    load_crt_vars=False,
    enable_cache=False,
    )
    
    signal_generator = hist.HistogramGenerator(
        rundata["mc"],
        binning.copy(),
#         data_pot=data_pot,
#         selection=selection,
#         preselection=preselection,
#         sideband_generator=None,
#         uncertainty_defaults=None,
        detvar_data=detvar_data,
        normalization_uncertainty=[0.01,0.02]
    )
    
    make_error_plot([signal_generator], "test_error_plot.pdf")
    plt.show()

Loading devar histograms from file: /exp/uboone/data/users/mmoudgal/PELEE/detvar_cached_dataframes//run_3_RecoDeltaPT_bnb.json


TypeError: 'HistogramGenerator' object is not iterable

In [ ]:
for binning_def in vdef.TKI_variables_1e1p:
    binning = hist.Binning.from_config(*binning_def)
    #print(binning.is_compatible(binning.copy()))
    print(type(binning.copy().variable))
    print()

In [ ]:
errors = [0.2, 0.8]
errors2 = [0.6, 0.4]
edges = [0, 80, 180]
plt.stairs(errors, edges, label='error 1', linestyle='dashdot')
plt.stairs(errors2, edges, label='error 2', lw=1.7)
plt.legend()
#plt.axis([0, 180, 0, 1.2]) 
#plt.axis([edges[0], edges[-1], 0, 1.2])